In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d",
    "jacobi1d",
    "fastwalshtransform",
    "relu",
    "fft",
    "fir",
    "sssp",
    "convolution2d",
    "color",
    "bitonicsort",
    "pagerank",
    "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'spmv',
    'mis', 
    'stencil2d',
    'gemm',
    'lu',
    'matrixmultiplication',
    'mm2',
    'mm3',
    'gaussian',
    'matrixtranspose',
    'im2col',
    'vgg16',
    'bfs',
]

In [7]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [5]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [9]:
import pandas as pd
import numpy as np
import os

# RO 구간 설정
bins = [1, 9, 18, 27, 36]
labels = ['2~9', '10~18', '19~27', '28~36']

results = []

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"Skip: No {trace_file}")
        continue

    print(f"Processing {benchmark}...")
    df = pd.read_csv(trace_file, skipinitialspace=True)

    # ===== addr별 집계 =====
    agg_dict = {col: 'sum' for col in chip_cols}
    agg_dict['rw'] = 'max'
    agg_dict['access_count'] = 'sum'
    grouped = df.groupby('addr').agg(agg_dict).reset_index()

    # 기본 계산
    shared_chiplet_counts = (grouped[chip_cols] > 0).sum(axis=1)
    shared_mask = shared_chiplet_counts > 1
    single_mask = shared_chiplet_counts == 1
    is_rw = grouped['rw'] == 1
    is_ro = ~is_rw

    total_cachelines = len(grouped)
    total_access = grouped['access_count'].sum()

    # ===== 1. Sharing Degree (RO/RW/Non-shared) =====
    ro_shared_acc = grouped.loc[shared_mask & is_ro, 'access_count'].sum()
    rw_shared_acc = grouped.loc[shared_mask & is_rw, 'access_count'].sum()
    non_shared_acc = grouped.loc[~shared_mask, 'access_count'].sum()

    ro_shared_cnt = (shared_mask & is_ro).sum()
    rw_shared_cnt = (shared_mask & is_rw).sum()

    # RO 구간별
    ro_shared_mask = shared_mask & is_ro
    ro_shared_counts = shared_chiplet_counts[ro_shared_mask]
    ro_bins = pd.cut(ro_shared_counts, bins=bins, labels=labels, right=True)
    ro_bin_cnt_ratios = ro_bins.value_counts(sort=False) / total_cachelines

    # ===== 2. True/False/Non-Sharing (RO/RW 모두) =====
    # True sharing (>=2 chiplet)
    true_ro_cnt = (shared_mask & is_ro).sum()
    true_rw_cnt = (shared_mask & is_rw).sum()
    true_ro_acc = grouped.loc[shared_mask & is_ro, 'access_count'].sum()
    true_rw_acc = grouped.loc[shared_mask & is_rw, 'access_count'].sum()
    true_total_cnt = true_ro_cnt + true_rw_cnt
    true_total_acc = true_ro_acc + true_rw_acc

    # False/Non sharing (1 chiplet)
    false_ro_cnt = false_rw_cnt = false_ro_acc = false_rw_acc = 0
    non_ro_cnt = non_rw_cnt = non_ro_acc = non_rw_acc = 0
    
    if single_mask.sum() > 0:
        single_grouped = grouped[single_mask].copy()
        chiplet_data = single_grouped[chip_cols].values
        owner_idx = single_grouped['owner'].astype(int).values
        owner_accessed = chiplet_data[np.arange(len(single_grouped)), owner_idx] > 0
        
        # False sharing (~owner_access)
        false_mask = ~owner_accessed
        false_ro_mask = false_mask & single_grouped['rw'] == 0
        false_rw_mask = false_mask & single_grouped['rw'] == 1
        false_ro_cnt = false_ro_mask.sum()
        false_rw_cnt = false_rw_mask.sum()
        false_ro_acc = single_grouped.loc[false_ro_mask, 'access_count'].sum()
        false_rw_acc = single_grouped.loc[false_rw_mask, 'access_count'].sum()
        
        # Non sharing (owner_access)
        non_mask = owner_accessed
        non_ro_mask = non_mask & single_grouped['rw'] == 0
        non_rw_mask = non_mask & single_grouped['rw'] == 1
        non_ro_cnt = non_ro_mask.sum()
        non_rw_cnt = non_rw_mask.sum()
        non_ro_acc = single_grouped.loc[non_ro_mask, 'access_count'].sum()
        non_rw_acc = single_grouped.loc[non_rw_mask, 'access_count'].sum()

    # ===== 결과 저장 (RO/RW 모두) =====
    result_row = {
        # Sharing Degree
        'total_cachelines': total_cachelines,
        'total_access_count': total_access,
        'ro_shared_cnt': ro_shared_cnt, 'rw_shared_cnt': rw_shared_cnt,
        'non_shared_cnt': total_cachelines - ro_shared_cnt - rw_shared_cnt,
        'ro_shared_cnt_ratio': ro_shared_cnt / total_cachelines,
        'rw_shared_cnt_ratio': rw_shared_cnt / total_cachelines,
        'ro_shared_acc_count': ro_shared_acc, 'rw_shared_acc_count': rw_shared_acc,
        'ro_shared_acc_ratio': ro_shared_acc / total_access,
        'rw_shared_acc_ratio': rw_shared_acc / total_access,
        **{f'ro_{lbl}_cnt_ratio': ro_bin_cnt_ratios.get(lbl, 0.0) for lbl in labels},

        # True Sharing (RO/RW)
        'true_ro_cnt': true_ro_cnt, 'true_rw_cnt': true_rw_cnt,
        'true_ro_cnt_ratio': true_ro_cnt / total_cachelines,
        'true_rw_cnt_ratio': true_rw_cnt / total_cachelines,
        'true_ro_acc_count': true_ro_acc, 'true_rw_acc_count': true_rw_acc,
        'true_ro_acc_ratio': true_ro_acc / total_access,
        'true_rw_acc_ratio': true_rw_acc / total_access,

        # False Sharing (RO/RW)
        'false_ro_cnt': false_ro_cnt, 'false_rw_cnt': false_rw_cnt,
        'false_ro_cnt_ratio': false_ro_cnt / total_cachelines,
        'false_rw_cnt_ratio': false_rw_cnt / total_cachelines,
        'false_ro_acc_count': false_ro_acc, 'false_rw_acc_count': false_rw_acc,
        'false_ro_acc_ratio': false_ro_acc / total_access,
        'false_rw_acc_ratio': false_rw_acc / total_access,

        # Non-Sharing (RO/RW)
        'non_ro_cnt': non_ro_cnt, 'non_rw_cnt': non_rw_cnt,
        'non_ro_cnt_ratio': non_ro_cnt / total_cachelines,
        'non_rw_cnt_ratio': non_rw_cnt / total_cachelines,
        'non_ro_acc_count': non_ro_acc, 'non_rw_acc_count': non_rw_acc,
        'non_ro_acc_ratio': non_ro_acc / total_access,
        'non_rw_acc_ratio': non_rw_acc / total_access,
    }
    result_row['benchmark'] = benchmark
    results.append(result_row)

# ===== 저장 & 출력 =====
if results:
    result_df = pd.DataFrame(results)
    output_dir = "./sharing-degree"
    os.makedirs(output_dir, exist_ok=True)
    result_df.to_csv(f"{output_dir}/sharing_degree_full.csv", index=False, float_format='%.12f')
    
    print("✅ Saved to sharing_degree_full.csv")
    
    # 요약 테이블
    summary_cols = ['benchmark', 'true_ro_cnt_ratio', 'true_rw_cnt_ratio', 
                    'false_ro_cnt_ratio', 'false_rw_cnt_ratio',
                    'non_ro_cnt_ratio', 'non_rw_cnt_ratio']
    print("\n📊 SUMMARY (cnt_ratio)")
    print(result_df[summary_cols].round(4))


Processing fdtd2d...
Processing jacobi1d...
Processing fastwalshtransform...
Processing relu...
Processing fft...
Processing fir...
Processing sssp...
Processing convolution2d...
Processing color...
Processing bitonicsort...
Processing pagerank...
Processing simpleconvolution...
Skip: No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223953/expanded/gups_mesh_6x6_memory_trace.csv
Processing syr2k...
Processing spmv...
Processing mis...
Processing stencil2d...
Processing gemm...
Processing lu...
Processing matrixmultiplication...
Processing mm2...
Processing mm3...
Processing gaussian...
Processing matrixtranspose...
Processing im2col...
Processing vgg16...
Processing bfs...
✅ Saved to complete_sharing_analysis.csv

📊 Sharing Degree (접근 횟수 비율):
               benchmark  ro_shared_acc_ratio  rw_shared_acc_ratio  \
0                 fdtd2d               0.0002               0.0071   
1               jacobi1d               0.0001               0.0000   
2     fastwalshtransform      